# Qwen Portable Slice Row-Processing Proof (Colab GPU)

This notebook is the first live Colab proof for the portable-slice Task 103 row-processing lane.

Invariants:
- Hemma remains the only place that selects rows.
- This notebook consumes one Hemma-issued portable slice bundle.
- The notebook is only an orchestrator around repo-owned script surfaces.
- Output must match the canonical Task 103 row-processing run-root contract.


## Hemma preparation (run before opening this notebook)

Use a fresh proof-only `source-selection` universe, not the live Hemma `10k` run:

```bash
pdm run run-hemma -- pdm run task-103-preprocess-public-corpus launch \
  --task103-stage source-selection \
  --launch-id task121-colab-proof-selection-launch-20260310a \
  --task103-run-id task121-colab-proof-selection-20260310a \
  --rixvox-split train \
  --rixvox-max-rows-per-split 512 \
  --skip-build

pdm run run-hemma -- pdm run task-121-colab-slice-bundle plan \
  --source-run-root /srv/scratch/sir-convert-a-lot/build/runs/qwen3-tts-swedish-preprocessing/task121-colab-proof-selection-20260310a \
  --output-root /srv/scratch/sir-convert-a-lot/build/reference/qwen3-tts-colab-slices/task121-proof-slice-1-of-2-20260310a \
  --slice-count 2 \
  --slice-index 1
```

Then make these three files available under `SLICE_ROOT` in Colab:
- `selected_source_records.jsonl`
- `required_hub_files.json`
- `slice_summary.json`


In [ ]:
%pip install -q huggingface_hub pyarrow soundfile librosa transformers python-dotenv


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import tarfile
import time

REPO_ROOT = Path.cwd()
SLICE_ROOT = REPO_ROOT / "colab_inputs" / "task121-proof-slice-1-of-2-20260310a"
PROOF_BUNDLE_PATH = REPO_ROOT / "colab_ml_training" / "proof_inputs" / "task122-proof-slice-1-of-2-20260310a-bundle.tar.gz"
DATA_ROOT = Path("/content/data/qwen3-tts-swedish-corpus")
RUN_ROOT = Path("/content/work/runs/task121-colab-proof-rowproc-20260310a")
OUTPUT_ROOT = Path("/content/work/reference/qwen3-tts-swedish-corpus")
CACHE_DIR = Path("/content/cache/huggingface")
ROW_WORKER_COUNT = 4
GPU_ASR_WORKER_COUNT = 1

assert (REPO_ROOT / "scripts" / "sir_convert_a_lot").exists(), (
    "Open this notebook from the repo root so the repo-owned scripts are importable."
)

for path in (SLICE_ROOT, DATA_ROOT, RUN_ROOT.parent, OUTPUT_ROOT.parent, CACHE_DIR):
    path.mkdir(parents=True, exist_ok=True)

print(json.dumps({
    "repo_root": REPO_ROOT.as_posix(),
    "slice_root": SLICE_ROOT.as_posix(),
    "proof_bundle_path": PROOF_BUNDLE_PATH.as_posix(),
    "data_root": DATA_ROOT.as_posix(),
    "run_root": RUN_ROOT.as_posix(),
    "output_root": OUTPUT_ROOT.as_posix(),
    "cache_dir": CACHE_DIR.as_posix(),
    "row_worker_count": ROW_WORKER_COUNT,
    "gpu_asr_worker_count": GPU_ASR_WORKER_COUNT,
}, indent=2))


In [ ]:
if not PROOF_BUNDLE_PATH.exists():
    raise FileNotFoundError(
        "Expected committed proof bundle at " + PROOF_BUNDLE_PATH.as_posix()
    )

with tarfile.open(PROOF_BUNDLE_PATH, "r:gz") as archive:
    archive.extractall(SLICE_ROOT)

required_bundle_files = [
    SLICE_ROOT / "selected_source_records.jsonl",
    SLICE_ROOT / "required_hub_files.json",
    SLICE_ROOT / "slice_summary.json",
]
missing_bundle_files = [path.as_posix() for path in required_bundle_files if not path.exists()]
if missing_bundle_files:
    raise FileNotFoundError("Portable slice bundle extraction failed: " + ", ".join(missing_bundle_files))

slice_summary = json.loads((SLICE_ROOT / "slice_summary.json").read_text(encoding="utf-8"))
slice_summary


## Stage only the required raw files for this slice

This uses the repo-owned portable-slice staging surface and only downloads the exact Hub files listed in `required_hub_files.json`.


In [ ]:
stage_command = [
    sys.executable,
    "-m",
    "scripts.sir_convert_a_lot.devops.task121_qwen_colab_slice_bundle",
    "stage-required-files",
    "--slice-root",
    str(SLICE_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--cache-dir",
    str(CACHE_DIR),
]
print(" ".join(stage_command))
subprocess.run(stage_command, check=True, cwd=REPO_ROOT)


## Run canonical Task 103 row-processing on the portable slice

This is the actual proof step. It must emit the same Task 103 run-root structure as Hemma row-processing while using the Colab GPU for a single ASR scorer slot.


In [ ]:
rowproc_command = [
    sys.executable,
    "-m",
    "scripts.sir_convert_a_lot.devops.run_task103_qwen_swedish_preprocessing",
    "--source-mode",
    "selected-source-records",
    "--selected-source-records-path",
    str(SLICE_ROOT / "selected_source_records.jsonl"),
    "--data-root",
    str(DATA_ROOT),
    "--run-root",
    str(RUN_ROOT),
    "--output-root",
    str(OUTPUT_ROOT),
    "--stage",
    "row-processing",
    "--row-worker-count",
    str(ROW_WORKER_COUNT),
    "--gpu-asr-worker-count",
    str(GPU_ASR_WORKER_COUNT),
]
print(" ".join(rowproc_command))
started_at = time.time()
subprocess.run(rowproc_command, check=True, cwd=REPO_ROOT)
print({"elapsed_seconds": round(time.time() - started_at, 2)})


In [ ]:
status_payload = json.loads((RUN_ROOT / "status.json").read_text(encoding="utf-8"))
run_payload = json.loads((RUN_ROOT / "run.json").read_text(encoding="utf-8"))
spool_count = sum(1 for _ in RUN_ROOT.rglob("spool/rows/**/*.json"))
audio_count = sum(1 for _ in RUN_ROOT.rglob("audio_24k/**/*.wav"))
print(json.dumps({
    "status": status_payload,
    "run": run_payload,
    "spool_rows": spool_count,
    "audio_24k_files": audio_count,
}, indent=2))


## Success criteria

A successful first Colab proof should show:
- valid `run.json`
- valid `status.json`
- non-empty `inventory/`
- non-empty `audio_24k/`
- non-empty `spool/rows/`
- no notebook-only preprocessing logic outside these repo-owned commands
